# Xử lý dữ liệu Sinh hiệu định kỳ (Vital Signs)

**Mục tiêu:** Trích xuất 9 đặc trưng sinh hiệu đại diện cho 24 giờ đầu tiên nhập ICU từ bảng `vitalPeriodic.csv`.

**Input:**
- `raw/eicu/vitalPeriodic.csv` — dữ liệu sinh hiệu gốc (5 phút/lần)
- `processed/patient_clean.csv` — danh sách bệnh nhân hợp lệ

**Output:** `processed/vital_clean.csv`

**9 đặc trưng đầu ra:**
- `heartrate_mean`, `heartrate_min`, `heartrate_max`
- `systemicmean_max` (MAP)
- `sao2_mean`, `sao2_min` (SpO2)
- `respiration_mean`, `respiration_max`
- `temperature_mean`

In [12]:
# ============================================================
# BƯỚC 1: KHỞI TẠO VÀ NẠP DỮ LIỆU
# ============================================================
# vitalPeriodic.csv là bảng LỚN NHẤT trong 4 bảng (~1.6 triệu dòng)
# vì ghi sinh hiệu tự động mỗi 5 phút.
# => Cần nạp cẩn thận, lọc cohort SỚM để tiết kiệm RAM.

import pandas as pd
import numpy as np

# Đường dẫn (tính từ data_processing/)
VITAL_PATH = "../data/raw/eicu/vitalPeriodic.csv"
PATIENT_PATH = "../data/processed/patient_clean.csv"
OUTPUT_PATH = "../data/processed/vital_clean.csv"

# Đọc danh sách bệnh nhân hợp lệ TRƯỚC (file nhỏ)
df_patient = pd.read_csv(PATIENT_PATH)
valid_ids = set(df_patient["patientunitstayid"])
print(f"Số bệnh nhân hợp lệ trong cohort: {len(valid_ids):,}")

# Đọc dữ liệu vital gốc
df_vital = pd.read_csv(VITAL_PATH)
print(df_vital.info())
print(f"\nKích thước vitalPeriodic gốc: {df_vital.shape}")
print("\n5 dòng đầu tiên:")
display(df_vital.head())

# Giải phóng bộ nhớ DataFrame gốc (không còn dùng)
del df_patient
import gc

gc.collect()
print("\nĐã giải phóng bộ nhớ.")

Số bệnh nhân hợp lệ trong cohort: 2,326
<class 'pandas.DataFrame'>
RangeIndex: 1634960 entries, 0 to 1634959
Data columns (total 19 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   vitalperiodicid    1634960 non-null  int64  
 1   patientunitstayid  1634960 non-null  int64  
 2   observationoffset  1634960 non-null  int64  
 3   temperature        112820 non-null   float64
 4   sao2               1441700 non-null  float64
 5   heartrate          1627894 non-null  float64
 6   respiration        1381351 non-null  float64
 7   cvp                202284 non-null   float64
 8   etco2              72780 non-null    float64
 9   systemicsystolic   226475 non-null   float64
 10  systemicdiastolic  226471 non-null   float64
 11  systemicmean       228400 non-null   float64
 12  pasystolic         32083 non-null    float64
 13  padiastolic        32080 non-null    float64
 14  pamean             32632 non-null    float64
 15  st1

,vitalperiodicid,patientunitstayid,observationoffset,temperature,sao2,heartrate,respiration,cvp,etco2,systemicsystolic,systemicdiastolic,systemicmean,pasystolic,padiastolic,pamean,st1,st2,st3,icp
0,29524122,141765,1179,NaN,NaN,82.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29557845,141765,189,NaN,97.0,76.0,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,29524442,141765,1169,NaN,NaN,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,29513052,141765,1534,NaN,NaN,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,29524600,141765,1164,NaN,NaN,86.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Đã giải phóng bộ nhớ.


In [13]:
# ============================================================
# BƯỚC 2: LỌC COHORT VÀ CHỌN CỘT
# ============================================================
# 1. Lọc chỉ giữ bệnh nhân thuộc cohort hợp lệ (giảm từ 1.6M xuống ~550K dòng)
# 2. Chỉ giữ 7 cột cần thiết cho 9 đặc trưng

n_before = len(df_vital)

# Lọc theo cohort
df_vital = df_vital[df_vital["patientunitstayid"].isin(valid_ids)].copy()
print(f"Sau khi lọc cohort: {len(df_vital):,} dòng "
      f"(giảm {n_before - len(df_vital):,} dòng)")

# Chọn 7 cột cần thiết
VITAL_COLS = [
    "patientunitstayid",  # Mã định danh đợt nằm ICU (Khóa chính liên kết bệnh nhân)
    "observationoffset",  # Mốc thời gian ghi nhận sinh hiệu (Số phút từ lúc nhập ICU = 0)
    "heartrate",  # Nhịp tim (bpm)
    "systemicmean",  # Huyết áp động mạch trung bình / MAP (mmHg)
    "sao2",  # Độ bão hòa oxy trong máu / SpO2 (%)
    "respiration",  # Nhịp thở (lần/phút)
    "temperature",  # Nhiệt độ cơ thể (°C)
]
df_vital = df_vital[VITAL_COLS].copy()

print(f"\nSau khi chọn cột: {df_vital.shape}")
print("\nThông tin kiểu dữ liệu:")
df_vital.info()

Sau khi lọc cohort: 1,612,139 dòng (giảm 22,821 dòng)

Sau khi chọn cột: (1612139, 7)

Thông tin kiểu dữ liệu:
<class 'pandas.DataFrame'>
Index: 1612139 entries, 0 to 1634959
Data columns (total 7 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   patientunitstayid  1612139 non-null  int64  
 1   observationoffset  1612139 non-null  int64  
 2   heartrate          1605180 non-null  float64
 3   systemicmean       226834 non-null   float64
 4   sao2               1421917 non-null  float64
 5   respiration        1360197 non-null  float64
 6   temperature        110590 non-null   float64
dtypes: float64(5), int64(2)
memory usage: 98.4 MB


In [14]:
# ============================================================
# BƯỚC 3: LỌC CỬA SỔ THỜI GIAN 24 GIỜ ĐẦU (TIME WINDOW)
# ============================================================
# Chỉ lấy các bản ghi trong 24h đầu tiên nhập ICU:
#   0 <= observationoffset <= 1440 phút (24h × 60 phút)
#
# Lý do: bài báo gốc chỉ dùng dữ liệu 24h đầu để dự đoán tử vong,
# vì đây là cửa sổ có nhiều thông tin tiên lượng nhất và
# tránh nhiễu từ các can thiệp điều trị sau đó.

n_before = len(df_vital)

df_vital = df_vital[
    (df_vital["observationoffset"] >= 0) &
    (df_vital["observationoffset"] <= 1440)
    ].copy()

n_after = len(df_vital)
print(f"Trước lọc 24h: {n_before:,} dòng")
print(f"Sau lọc 24h  : {n_after:,} dòng")
print(f"Đã loại bỏ   : {n_before - n_after:,} dòng "
      f"({(1 - n_after / n_before) * 100:.2f}%)")

# Kiểm tra phân bố offset
print(f"\nobservationoffset: min={df_vital['observationoffset'].min()} "
      f"max={df_vital['observationoffset'].max()}")

# Số bản ghi trung bình mỗi bệnh nhân trong 24h
records_per_patient = df_vital.groupby("patientunitstayid").size()
print(f"\nSố bản ghi/bệnh nhân trong 24h:")
print(f"  mean={records_per_patient.mean():.1f}  "
      f"median={records_per_patient.median():.0f}  "
      f"max={records_per_patient.max()}")
print(f"  (Lý thuyết tối đa: 1440/5 = 288 bản ghi nếu đo liên tục)")

Trước lọc 24h: 1,612,139 dòng
Sau lọc 24h  : 543,699 dòng
Đã loại bỏ   : 1,068,440 dòng (66.27%)

observationoffset: min=0 max=1440

Số bản ghi/bệnh nhân trong 24h:
  mean=242.7  median=277  max=289
  (Lý thuyết tối đa: 1440/5 = 288 bản ghi nếu đo liên tục)


In [16]:
# ============================================================
# BƯỚC 4: XỬ LÝ NHIỄU VÀ GIÁ TRỊ BẤT THƯỜNG (DATA CLEANING)
# ============================================================
# Áp dụng quy tắc làm sạch chuẩn lâm sàng theo bài báo gốc:
#   1. systemicmean (MAP): loại bỏ < 20 mmHg (lỗi thiết bị)
#   2. temperature: giá trị > 45 là độ F -> chuyển sang °C
#   3. respiration: loại bỏ < 5 lần/phút

print("Thống kê TRƯỚC khi làm sạch:")
print(df_vital[["heartrate", "systemicmean", "sao2",
                "respiration", "temperature"]].describe())

# ---------- 1. MAP: loại bỏ < 20 mmHg ----------
n_bad_map = (df_vital["systemicmean"] < 20).sum()
print(f"\n1. systemicmean < 20 mmHg: {n_bad_map:,} dòng -> NaN")
df_vital.loc[df_vital["systemicmean"] < 20, "systemicmean"] = np.nan

# ---------- 2. Temperature: > 45 là độ F -> đổi sang °C ----------
mask_fahrenheit = df_vital["temperature"] > 45
n_fahrenheit = mask_fahrenheit.sum()
print(f"\n2. temperature > 45 (độ F): {n_fahrenheit:,} dòng -> chuyển sang °C")
df_vital.loc[mask_fahrenheit, "temperature"] = (
        (df_vital.loc[mask_fahrenheit, "temperature"] - 32) * 5 / 9
)

# Sau khi chuyển, vẫn còn giá trị bất thường (< 30 hoặc > 45) -> NaN
n_bad_temp = (
        (df_vital["temperature"] < 30) | (df_vital["temperature"] > 45)
).sum()
print(f"   Sau chuyển đổi, temperature ngoài [30, 45]: {n_bad_temp:,} dòng -> NaN")
df_vital.loc[
    (df_vital["temperature"] < 30) | (df_vital["temperature"] > 45),
    "temperature"
] = np.nan

# ---------- 3. Respiration: loại bỏ < 5 lần/phút ----------
n_bad_resp = (df_vital["respiration"] < 5).sum()
print(f"\n3. respiration (hô hấp) < 5 lần/phút: {n_bad_resp:,} dòng -> NaN")
df_vital.loc[df_vital["respiration"] < 5, "respiration"] = np.nan

# ---------- 4. Heartrate: loại bỏ <= 0 (bổ sung) ----------
n_bad_hr = ((df_vital["heartrate"] <= 0) | (df_vital["heartrate"] > 300)).sum()
print(f"\n4. heartrate <= 0: {n_bad_hr:,} dòng -> NaN")
df_vital.loc[df_vital["heartrate"] <= 0, "heartrate"] = np.nan

# ---------- 5. Sao2: loại bỏ <= 0 hoặc > 100 (bổ sung) ----------
n_bad_sao2 = ((df_vital["sao2"] <= 0) | (df_vital["sao2"] > 100)).sum()
print(f"\n5. sao2 ngoài (0, 100]: {n_bad_sao2:,} dòng -> NaN")
df_vital.loc[
    (df_vital["sao2"] <= 0) | (df_vital["sao2"] > 100), "sao2"
] = np.nan

print("\nThống kê SAU khi làm sạch:")
print(df_vital[["heartrate", "systemicmean", "sao2",
                "respiration", "temperature"]].describe())

Thống kê TRƯỚC khi làm sạch:
           heartrate  systemicmean           sao2    respiration   temperature
count  540954.000000  75629.000000  479603.000000  472901.000000  29549.000000
mean       84.277576     79.944003      96.496694      19.672597     37.054287
std        18.974507     19.830444       5.004650       6.272784      1.117263
min         0.000000     20.000000       0.000000       5.000000     30.050000
25%        70.000000     68.000000      95.000000      16.000000     36.600000
50%        83.000000     77.000000      97.000000      19.000000     37.200000
75%        96.000000     88.000000      99.000000      23.000000     37.700000
max       202.000000    346.000000     100.000000     163.000000     42.900000

1. systemicmean < 20 mmHg: 0 dòng -> NaN

2. temperature > 45 (độ F): 0 dòng -> chuyển sang °C
   Sau chuyển đổi, temperature ngoài [30, 45]: 0 dòng -> NaN

3. respiration (hô hấp) < 5 lần/phút: 0 dòng -> NaN

4. heartrate <= 0: 126 dòng -> NaN

5. sao2 ngoài

In [17]:
# ============================================================
# BƯỚC 5: GROUPBY VÀ TRÍCH XUẤT 9 ĐẶC TRƯNG
# ============================================================
# Gom nhóm theo patientunitstayid, tính 9 đặc trưng đại diện 24h đầu.
#
# Lưu ý: pandas .agg() tự động bỏ qua NaN, nên nếu bệnh nhân chỉ có
# 1 giá trị heartrate hợp lệ, mean/min/max đều = giá trị đó.

print(f"Số bệnh nhân có dữ liệu vital: {df_vital['patientunitstayid'].nunique():,}")

# Groupby và aggregate
vital_features = df_vital.groupby("patientunitstayid").agg(
    # Heartrate: mean, min, max
    heartrate_mean=("heartrate", "mean"),
    heartrate_min=("heartrate", "min"),
    heartrate_max=("heartrate", "max"),
    # Systemicmean (MAP): max
    systemicmean_max=("systemicmean", "max"),
    # Sao2: mean, min
    sao2_mean=("sao2", "mean"),
    sao2_min=("sao2", "min"),
    # Respiration: mean, max
    respiration_mean=("respiration", "mean"),
    respiration_max=("respiration", "max"),
    # Temperature: mean
    temperature_mean=("temperature", "mean"),
).reset_index()

print(f"\nKích thước bảng đặc trưng: {vital_features.shape}")
print("\n5 dòng đầu tiên:")
print(vital_features.head())

print("\nThống kê 9 đặc trưng:")
print(vital_features.describe().round(2))

Số bệnh nhân có dữ liệu vital: 2,240

Kích thước bảng đặc trưng: (2240, 10)

5 dòng đầu tiên:
   patientunitstayid  heartrate_mean  heartrate_min  heartrate_max  \
0             141764      106.652174           90.0          138.0   
1             141765       83.190972           72.0          116.0   
2             143870       45.449367           40.0           55.0   
3             144815       83.800000           68.0          118.0   
4             145427       65.457983           49.0           85.0   

   systemicmean_max  sao2_mean  sao2_min  respiration_mean  respiration_max  \
0               NaN        NaN       NaN               NaN              NaN   
1               NaN  96.608696      95.0         24.782609             39.0   
2              79.0  96.284810      80.0         67.031646             86.0   
3               NaN  98.630682      96.0         16.927273             42.0   
4               NaN  95.761905      89.0         16.349206             41.0   

   tempera

In [18]:
# ============================================================
# BƯỚC 6: KIỂM TRA VÀ XUẤT FILE
# ============================================================

# ---------- Kiểm tra tính duy nhất ----------
n_dup = vital_features["patientunitstayid"].duplicated().sum()
print(f"Số patientunitstayid trùng lặp: {n_dup}")
assert n_dup == 0, "Có trùng lặp!"

# ---------- Tỷ lệ NaN của 9 đặc trưng ----------
print(f"\nTổng số bệnh nhân có dữ liệu vital: {len(vital_features):,}")
print("\nTỷ lệ NaN của 9 đặc trưng:")
feature_cols = [c for c in vital_features.columns if c != "patientunitstayid"]
for col in feature_cols:
    n_miss = vital_features[col].isna().sum()
    pct = n_miss / len(vital_features) * 100
    print(f"  {col:20s}: {n_miss:>4,} ({pct:>5.2f}%)")

# ---------- Lưu file ----------
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
vital_features.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Đã lưu: {OUTPUT_PATH}")
print(f"   Kích thước cuối: {vital_features.shape}")

Số patientunitstayid trùng lặp: 0

Tổng số bệnh nhân có dữ liệu vital: 2,240

Tỷ lệ NaN của 9 đặc trưng:
  heartrate_mean      :   11 ( 0.49%)
  heartrate_min       :   11 ( 0.49%)
  heartrate_max       :   11 ( 0.49%)
  systemicmean_max    : 1,855 (82.81%)
  sao2_mean           :   36 ( 1.61%)
  sao2_min            :   36 ( 1.61%)
  respiration_mean    :  205 ( 9.15%)
  respiration_max     :  205 ( 9.15%)
  temperature_mean    : 2,095 (93.53%)

✅ Đã lưu: ../data/processed/vital_clean.csv
   Kích thước cuối: (2240, 10)
